# P1-4 · Học không giám sát và feature engineering

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q seaborn

## Lý thuyết

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

# kiểm tra ví dụ 6 điểm ở trên, với đúng hai tâm khởi tạo kém 1 và 3
x6 = np.array([[1], [3], [5], [20], [22], [24]])
km6 = KMeans(n_clusters=2, init=np.array([[1.0], [3.0]]), n_init=1).fit(x6)
print(km6.labels_, km6.cluster_centers_.ravel(), km6.n_iter_)   # tâm 3 và 22

# 400 "khách hàng" giả lập có 4 nhóm, 2 đặc trưng đã chuẩn hóa
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
X, _ = make_blobs(n_samples=400, centers=4, cluster_std=1.2, random_state=0)
X_scaled = StandardScaler().fit_transform(X)
km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X_scaled)
print(km.labels_[:12])                   # cụm của từng điểm
print(km.cluster_centers_.round(2))      # tọa độ tâm (trên thang đã chuẩn hóa)
print(round(km.inertia_, 1))

[0 0 0 1 1 1] [ 3. 22.] 3
[1 3 3 1 3 0 2 3 3 2 0 2]
[[-0.68  1.36]
 [ 0.39  0.09]
 [ 1.13 -1.15]
 [-1.11 -0.48]]
186.9


In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
X_digits = load_digits().data             # 1797 ảnh 8×8 = 64 chiều
X_scaled = StandardScaler().fit_transform(X_digits)   # chuẩn hóa như trong hình ở trên
pca = PCA(n_components=0.95)              # giữ đủ thành phần để đạt 95% phương sai
Z = pca.fit_transform(X_scaled)
pca.n_components_, pca.explained_variance_ratio_[:3].round(3)

(40, array([0.12 , 0.096, 0.084]))


In [ ]:
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

num = ["tuổi", "thu_nhập"]; cat = ["thành_phố", "kênh"]
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                      ("sc", StandardScaler())]), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat),
])
model = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])

# dữ liệu khách hàng giả lập, có ô trống
rng = np.random.default_rng(0); n = 800
df = pd.DataFrame({"tuổi": rng.integers(18, 65, n).astype(float), "thu_nhập": rng.gamma(2, 8, n).round(1),
                   "thành_phố": rng.choice(["HN", "HCM", "ĐN"], n), "kênh": rng.choice(["web", "app", "zalo"], n)})
y = ((df["thu_nhập"] > 15) & (df["kênh"] != "zalo")).astype(int) ^ (rng.random(n) < 0.1)
df.loc[rng.random(n) < 0.1, "thu_nhập"] = np.nan; df.loc[rng.random(n) < 0.05, "thành_phố"] = np.nan
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.25, random_state=0)

model.fit(X_train, y_train)                 # một lệnh, không rò rỉ
cross_val_score(model, X_train, y_train).round(3)    # mỗi fold tự fit lại toàn bộ tiền xử lý

array([0.875, 0.833, 0.858, 0.875, 0.858])


## Bài tập 5: Pipeline hoàn chỉnh {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

df = sns.load_dataset("titanic")
X, y = df[["age", "fare", "sex", "embarked", "class"]], df["survived"]
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), ["age", "fare"]),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), ["sex", "embarked", "class"]),
])
pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
print("Pipeline CV:", cross_val_score(pipe, X, y, cv=5).mean().round(4))
Xall = pre.fit_transform(X)   # sai: học trên toàn bộ dữ liệu trước khi CV
print("Rò rỉ CV:", cross_val_score(LogisticRegression(max_iter=1000), Xall, y, cv=5).mean().round(4))

Pipeline CV: 0.789
Rò rỉ CV: 0.789
